# 🔥 Flambée

**Touche le bouton ▶︎ sur la cellule grise, juste en dessous.**
Puis attends une à deux minutes. Tout le reste est expliqué plus bas.


In [ ]:
#@title 🔥 Démarrer Flambée { display-mode: "form" }
#@markdown Touche ▶︎ puis attends l'encadré avec l'adresse.
MOT_DE_PASSE = ""  #@param {type:"string"}
#@markdown *(laisse vide pour qu'un mot de passe soit engendré)*
CLE_API_ANTHROPIC = ""  #@param {type:"string"}
#@markdown *(facultatif : pour générer le script en un clic. Sans clé, l'app*
#@markdown *affiche un prompt à copier dans une conversation Claude.)*
TRANSCRIPTION = True  #@param {type:"boolean"}
#@markdown *(active Script Viral et Voice Studio ; ajoute environ une minute au*
#@markdown *premier démarrage, puis c'est en cache)*

import subprocess, sys, os
from pathlib import Path

DOSSIER = Path("/content/flambee")
DEPOT = "https://github.com/bastienroels-ops/projet.git"
BRANCHE = "claude/fastapi-viral-video-montage-e9x7i8"
PORT = 8000

# Les vidéos produites vivent dans /content/flambee-data, hors du dossier
# cloné : elles survivent à une relance de cette cellule.

# 1. Récupération du code (clone la première fois, mise à jour ensuite).
if DOSSIER.exists():
    subprocess.run(["git", "-C", str(DOSSIER), "fetch", "--quiet", "origin", BRANCHE], check=False)
    subprocess.run(["git", "-C", str(DOSSIER), "checkout", "--quiet", BRANCHE], check=False)
    subprocess.run(["git", "-C", str(DOSSIER), "reset", "--hard", "--quiet", f"origin/{BRANCHE}"], check=False)
else:
    print("→ Téléchargement de Flambée…", flush=True)
    subprocess.run(["git", "clone", "--quiet", "--branch", BRANCHE, "--depth", "1",
                    DEPOT, str(DOSSIER)], check=True)

os.chdir(DOSSIER)
sys.path.insert(0, str(DOSSIER))

# 2. Installation puis démarrage du serveur et du tunnel.
#    Le moteur de transcription s'installe au passage, et son installation est
#    vérifiée par un import réel : pip peut réussir en laissant le module
#    inutilisable. En cas d'échec, la raison s'affiche ici plutôt que de se
#    transformer en « moteur absent » inexpliqué dans l'application.
from colab import launch

serveur, tunnel, adresse, mot_de_passe = launch.start_all(
    port=PORT, password=MOT_DE_PASSE.strip(),
    anthropic_key=CLE_API_ANTHROPIC.strip(),
    transcription=TRANSCRIPTION,
)

# 3. Lien de secours propre à Colab : il ne dépend d'aucun tunnel et
#    fonctionne dans le navigateur où tu es connecté à Google.
secours = None
try:
    from google.colab.output import eval_js
    secours = eval_js(f"google.colab.kernel.proxyPort({PORT})")
except Exception:
    pass

if adresse:
    print(launch.banner(adresse, "flambee", mot_de_passe))
else:
    print("\n⚠️  Le tunnel n'a pas pu s'ouvrir — utilise le lien de secours.")
    print(f"   Identifiant : flambee — Mot de passe : {mot_de_passe}\n")

if secours:
    print(f"  Lien de secours Colab : {secours}\n")

# 4. Cette boucle garde la session éveillée. Pour tout arrêter : bouton ⏹.
launch.keep_alive(serveur, tunnel, port=PORT, password=mot_de_passe)


## Ce qui se passe ensuite

1. L'installation défile pendant une à deux minutes.
2. Un encadré affiche une **adresse**, un **identifiant** et un **mot de passe**.
3. Ouvre l'adresse dans un nouvel onglet Safari et saisis les identifiants.
   Tu arrives **directement dans l'atelier** : pas de second formulaire.
4. *Partager → Sur l'écran d'accueil* pour l'avoir comme une app.

Le mot de passe ne change plus d'une relance à l'autre, tant que la machine
Colab vit : Safari peut donc le retenir. Pour qu'il soit le même d'une session
à la suivante, saisis-en un dans la cellule — il a priorité.

## Ce qu'il faut savoir

- **Laisse cet onglet Colab ouvert.** C'est lui qui fait tourner le serveur ;
  si tu le fermes, l'app s'arrête.
- **Télécharge tes vidéos avant la fin de la session.** Google récupère la
  machine après quelques heures (ou ~90 min d'inactivité) et tout ce qui est
  dessus disparaît.
- **Les liens TikTok/YouTube échoueront probablement.** Depuis un serveur
  Google, les plateformes réclament une connexion (« Sign in to confirm you're
  not a bot »). Utilise à la place le bouton **« Choisir des vidéos »** de
  l'étape 1 : enregistre d'abord les vidéos sur ton iPhone (TikTok propose
  *Enregistrer la vidéo* quand l'auteur l'autorise), puis importe-les.
- **Le rendu sera plus lent** que sur une vraie machine : commence toujours par
  l'**aperçu 540p**, et ne lance le rendu final qu'une fois le montage validé.
- Chaque démarrage donne une **nouvelle adresse**.
- **Si la page affiche une erreur 530 ou 1033**, c'est le tunnel qui a lâché,
  pas le rendu : reviens sur cet onglet, une nouvelle adresse y est affichée.
  Les vidéos déjà produites sont conservées dans `/content/flambee-data`.
- **Le script se rédige à la main par défaut.** À l'étape 4, le bouton
  *Écrire le script avec Claude* affiche un prompt : copie-le, colle-le dans
  une conversation Claude, et rapporte le texte obtenu dans le champ *Script*.
  Si tu as une clé API Anthropic, renseigne-la dans la cellule et la génération
  se fera en un clic.


## Quand tu as fini

Télécharge la vidéo depuis l'étape 5 (*Télécharger le .mp4*) : elle arrive dans
**Fichiers** sur ton iPhone, puis *Partager → Enregistrer dans Photos*.

Ensuite tu peux fermer l'onglet. Au prochain usage, rouvre ce carnet et touche
à nouveau ▶︎ : le code se met à jour tout seul.
